In [ ]:
import base64
import gzip
import json
import pickle
import pyrender
import numpy as np
import random
import secrets
import trimesh
import voxeloo
import zlib

from PIL import Image
from dataclasses import dataclass
from functools import cache
from typing import List, Optional, Dict, Tuple
from scipy import ndimage

### Pipeline input

In [ ]:
CHUNK_DIM = (128, 512, 128) # z, y, x
CHUNK_ORIGIN = (0, 0, 0)    # z, y, x
CHUNK_PADDING = 32
CHUNK_RNG = np.random.default_rng([secrets.randbits(32) for _ in range(624)])

In [ ]:
MAP_DIM = (
    CHUNK_DIM[0] + 2 * CHUNK_PADDING,
    CHUNK_DIM[1] + 2 * CHUNK_PADDING,
    CHUNK_DIM[2] + 2 * CHUNK_PADDING,
)
MAP_ORIGIN = (
    CHUNK_ORIGIN[0] - CHUNK_PADDING,
    CHUNK_ORIGIN[1] - CHUNK_PADDING,
    CHUNK_ORIGIN[2] - CHUNK_PADDING,
)

### Visualization routines

In [ ]:
def heatmap_to_image(heat):
    return Image.fromarray(np.uint8(heat * 255) , "L")

In [ ]:
def visualize_voxels(voxels, wireframe=False):
    if voxels.dtype == bool:
        voxels = np.array([0, 0xffffffff])[voxels.astype(int)]
    
    mesh = voxeloo.voxels.voxels_to_mesh(voxels)
    
    # Convert the mesh into the trimesh format.
    tm = trimesh.Trimesh(
        vertices=mesh.vertices[:, 0:3],
        faces=mesh.triangles,
    )
    
    # Add the vertex colors.
    tm.visual.face_colors = mesh.vertices[mesh.triangles[:, 0], 3:6]
    
    scene = pyrender.Scene(ambient_light=[0.1, 0.1, 0.1, 1.0])
    scene.add(
        pyrender.Mesh.from_trimesh(
            tm,
            smooth=False,
            wireframe=wireframe,
        )
    )
    pyrender.Viewer(
        scene,
        use_raymond_lighting=True,
    )

### Noise routines

In [ ]:
def seed_hash(key):
    return np.uint32(zlib.adler32(str(key).encode("utf-8")))

In [ ]:
def domain(shape, scale):
    assert len(shape) in [2, 3]
    coords = np.meshgrid(
        *[np.arange(dim) for dim in shape],
        indexing="ij",
    )
    if len(shape) == 2:
        coords[0] += MAP_ORIGIN[0]
        coords[1] += MAP_ORIGIN[1]
    else:
        coords[0] += MAP_ORIGIN[0]
        coords[1] += MAP_ORIGIN[1]
        coords[2] += MAP_ORIGIN[2]     
    return tuple(coord / scale for coord in coords)[::-1]

In [ ]:
@cache 
def gen(seed=None):
    return voxeloo.noise.SimplexNoise(seed_hash(seed or 1))
    
def noise(coords, seed=None):
    return gen(seed).noise(coords.reshape(-1, coords.shape[-1])).reshape(*coords.shape[0:-1])

In [ ]:
def fractal_noise(shape, period, octaves, persistence=0.5, lacunarity=2.0, transform=None, seed=None):
    ret = np.zeros(shape, dtype=float)
    for i in range(octaves):
        coords = np.stack(domain(shape, period / lacunarity**i), axis=-1)
        if transform:
            coords = transform(coords)
        ret += persistence**i * noise(coords, seed)
    return ret

In [ ]:
def blur(tensor, sigma):
    return ndimage.gaussian_filter(tensor.astype(float), sigma)

In [ ]:
%%time 

x = fractal_noise((1024, 1024), 512, 5, persistence=0.45, lacunarity=2.0)
x = (np.abs(x) < 0.005).astype(float) 
heatmap_to_image((x - x.min()) / (x.max() - x.min()))

### Cave map definitions

In [ ]:
@dataclass
class CaveMap:
    ids: Dict[str, int]
    assignments: np.ndarray

In [ ]:
def add_crevices(m: CaveMap):
    shape = m.assignments.shape
    
    period = 128
    octaves = 3
    persistence = 0.7
    
    def scale_y(coords):
        a = coords * [1, 4, 1]
        b = coords ** [1, 1.25, 1]
        return np.minimum(a, b)

    a = fractal_noise(
        shape,
        period,
        octaves,
        persistence,
        seed="a",
    )

    b = fractal_noise(
        shape,
        period,
        octaves,
        persistence,
        seed="b",
    )
    
    scale = np.clip(1 - domain(shape, 32.0)[1], 0.5, 1.0)
    
    a = scale * np.abs(a) 
    b = np.abs(b)
    
    mask = np.logical_and(a < 0.1, b < 0.1)
    
    # Limit to a certain depth
    nm = fractal_noise(shape, 128, 4, 0.4)
    mask = np.logical_and(
        mask,
        64 * nm + domain(shape, 1)[1] < 288,
    )
    
    i = m.ids.setdefault("cave", len(m.ids))
    m.assignments[mask] = i

In [ ]:
def add_tunnels(m: CaveMap):
    shape = m.assignments.shape
    
    period = 128
    octaves = 5
    persistence = 0.4
    
    def scale_y(coords):
        return coords * [1, 4, 1]

    a = fractal_noise(
        shape,
        period,
        octaves,
        persistence,
        transform=scale_y,
        seed="a",
    )

    b = fractal_noise(
        shape,
        period,
        octaves,
        persistence,
        transform=scale_y,
        seed="b",
    )
    
    mask = np.logical_and(np.abs(a) < 0.03, np.abs(b) < 0.03)
    mask = blur(mask, 3) > 0.01
    
    # Limit to a certain depth
    nm = fractal_noise(shape, 128, 4, 0.4)
    mask = np.logical_and(
        mask,
        np.logical_and(
            64 * nm + domain(shape, 1)[1] > 272,
            32 * nm + domain(shape, 1)[1] < 400,
        )
    )
    
    i = m.ids.setdefault("cave", len(m.ids))
    m.assignments[mask] = i

In [ ]:
def add_caverns(m: CaveMap):
    shape = m.assignments.shape
    
    period = 512
    octaves = 5
    persistence = 0.4
    
    def scale_y(coords):
        return coords * [1, 8, 1]

    a = fractal_noise(
        shape,
        period,
        octaves,
        persistence,
        transform=scale_y,
        seed="a",
    )

    b = fractal_noise(
        shape,
        period,
        octaves,
        persistence,
        transform=scale_y,
        seed="b",
    )
    
    mask = np.logical_and(np.abs(a) < 0.02, np.abs(b) < 0.02)
    mask = blur(mask, 4) > 0.01
    
    # Limit to a certain depth
    nm = fractal_noise(shape, 128, 4, 0.4)
    mask = np.logical_and(
        mask,
        32 * nm + domain(shape, 1)[1] > 320,
    )
    
    i = m.ids.setdefault("cave", len(m.ids))
    m.assignments[mask] = i

### Generate the cave map

In [ ]:
cm = CaveMap(
    ids={"ground": 0},
    assignments=np.zeros(shape=MAP_DIM, dtype=int),
)

In [ ]:
%%time

add_crevices(cm)

In [ ]:
%%time

add_tunnels(cm)

In [ ]:
%%time

add_caverns(cm)

In [ ]:
visualize_voxels(cm.assignments == cm.ids["cave"])

### Mineral map definitions

In [ ]:
@dataclass
class MineralMap:
    ids: Dict[str, int]
    colors: List[int]
    assignments: np.ndarray

In [ ]:
def add_basalt(m: MineralMap):
    shape = m.assignments.shape
    
    depth = 256
    scale = 0.1

    nm = fractal_noise(shape, period=128, octaves=3, persistence=0.4, seed="basalt")
    dm = scale * np.abs(domain(shape, 1.0)[1] - depth)

    mask = (nm - 0.1 * dm) > -0.2
    
    i = m.ids.setdefault("basalt", len(m.ids))
    m.assignments[mask] = i
    m.colors.append(0x3e4852ff)

In [ ]:
def add_limestone(m: MineralMap):
    shape = m.assignments.shape
    
    depth = 384
    scale = 0.001

    nm = fractal_noise(shape, period=256, octaves=3, persistence=0.6, seed="limestone")
    dm = scale * np.abs(domain(shape, 1.0)[1] - depth)

    mask = (nm - dm) > 0.12
    mask[m.assignments != 0] = False
    
    i = m.ids.setdefault("limestone", len(m.ids))
    m.assignments[mask] = i
    m.colors.append(0xc3c4b1ff)

In [ ]:
def add_granite(m: MineralMap):
    shape = m.assignments.shape
    
    depth = 448
    scale = 0.01

    nm = fractal_noise(shape, period=128, octaves=3, persistence=0.4, seed="granite")
    dm = scale * np.abs(domain(shape, 1.0)[1] - depth)

    mask = (nm - dm) > -0.3
    mask[m.assignments != 0] = False
    
    i = m.ids.setdefault("granite", len(m.ids))
    m.assignments[mask] = i
    m.colors.append(0xd7e0e0ff)

In [ ]:
def add_quartzite(m: MineralMap):
    shape = m.assignments.shape
    
    depth = 480
    scale = 0.001

    nm = fractal_noise(shape, period=128, octaves=3, persistence=0.6, seed="quartzite")
    dm = scale * np.abs(domain(shape, 1.0)[1] - depth)

    mask = (nm - dm) > 0.1
    mask[m.assignments != 0] = False
    
    i = m.ids.setdefault("quartzite", len(m.ids))
    m.assignments[mask] = i
    m.colors.append(0xa89c8aff)

In [ ]:
def add_bedrock(m: MineralMap):
    shape = m.assignments.shape
    
    depth = 512 - 16
    scale = 0.1

    nm = 0.5 * fractal_noise(shape, period=64, octaves=2, persistence=0.4, seed="bedrock")
    dm = scale * (domain(shape, 1.0)[1] - depth)

    mask = (nm + dm) > 0.1
    mask[:, -1, :] = True
    
    i = m.ids.setdefault("bedrock", len(m.ids))
    m.assignments[mask] = i
    m.colors.append(0x4e4a54ff)

### Generate the mineral map

In [ ]:
mm = MineralMap(
    ids={"stone": 0},
    assignments=np.zeros(shape=MAP_DIM, dtype=int),
    colors=[0xbebebeff],
)

In [ ]:
%%time 

add_basalt(mm)

In [ ]:
%%time 

add_granite(mm)

In [ ]:
%%time 

add_limestone(mm)

In [ ]:
%%time 

add_quartzite(mm)

In [ ]:
%%time 

add_bedrock(mm)

In [ ]:
visualize_voxels(np.array(mm.colors)[mm.assignments])

In [ ]:
for name, i in mm.ids.items():
    print(f"{name}: {(mm.assignments == i).mean()})")

### Generate the ore map

In [ ]:
@dataclass
class OreMap:
    ids: Dict[str, int]
    colors: List[int]
    assignments: np.ndarray

@dataclass
class Ore:
    name: str
    color: int
        
@dataclass
class Lode:
    name: str
    mask: np.ndarray
    prob: float

In [ ]:
def lode_mask(d, n, seed):
    assert d <= 4
    ret = np.zeros(shape=d * d * d)
    ret[:n] = 1
    np.random.seed(seed_hash(seed))
    np.random.shuffle(ret)
    return ret.reshape(d, d, d)
    
lodes = []

# Define some coal deposits.
for n in [4, 6, 8, 16]:
    lodes.append(
        Lode(
            name="coal_ore",
            mask=lode_mask(3, n, seed=f"coal_{n}"),
            prob=3e-3,
        )
    )
    
# Define some silver deposits.
for n in [4, 8, 16, 24]:
    lodes.append(
        Lode(
            name="silver_ore",
            mask=lode_mask(4, n, seed=f"silver_{n}"),
            prob=2e-3,
        )
    )
    
# Define some gold deposits.
for n in [4, 8, 12, 16]:
    lodes.append(
        Lode(
            name="gold_ore",
            mask=lode_mask(3, n, seed=f"gold_{n}"),
            prob=5e-4,
        )
    )
    
# Define some diamond deposits.
for n in [2, 3, 6, 8]:
    lodes.append(
        Lode(
            name="diamond_ore",
            mask=lode_mask(2, n, seed=f"diamond_{n}"),
            prob=2e-4,
        )
    )
    
    
# Define some neptunium deposits.
for n in [3, 4, 5, 6]:
    lodes.append(
        Lode(
            name="neptunium_ore",
            mask=lode_mask(3, n, seed=f"neptunium_{n}"),
            prob=1e-4,
        )
    )


In [ ]:
ores = [
    Ore("coal_ore", 0x282e2aff),
    Ore("silver_ore", 0xd8e3e3ff),
    Ore("gold_ore", 0xdecb52ff),
    Ore("diamond_ore", 0x89c5d9ff),
    Ore("neptunium_ore", 0x1d4030ff),
]

In [ ]:
om = OreMap(
    ids={"none": 0},
    colors=[0x00000000],
    assignments=np.zeros(shape=CHUNK_DIM, dtype=int),
)

for i, ore in enumerate(ores):
    om.ids[ore.name] = len(om.ids)
    om.colors.append(ore.color)
    
ranges = np.cumsum([lode.prob for lode in lodes])

In [ ]:
assert all(dim % 4 == 0 for dim in om.assignments.shape)


prob = CHUNK_RNG.random(tuple(dim // 4 for dim in om.assignments.shape))

# Assign ores to voxels.
for i, lode in enumerate(lodes):
    mask = prob < ranges[i]
    
    for ijk in np.stack(np.where(mask), axis=-1):
        lo = 4 * ijk
        hi = lo + lode.mask.shape
        om.assignments[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]] = om.ids[lode.name] * lode.mask
        
    prob[mask] = 1.0

In [ ]:
visualize_voxels(np.array(om.colors)[om.assignments])

### Generate the undeground map

In [ ]:
@dataclass
class UndegroundMap:
    ids: Dict[str, int]
    colors: List[int]
    assignments: np.ndarray

In [ ]:
um = UndegroundMap(
    ids={"none": 0},
    colors=[0x00000000],
    assignments=np.zeros(shape=CHUNK_DIM, dtype=int)
)

for name, color in zip([*mm.ids, *om.ids], [*mm.colors, *om.colors]):
    um.ids.setdefault(name, len(um.ids))
    if len(um.ids) > len(um.colors):
        um.colors.append(color)

#### Add minerals

In [ ]:
cp = CHUNK_PADDING
assignments = mm.assignments[cp:-cp, cp:-cp, cp:-cp]

for name, i in mm.ids.items():
    um.assignments[assignments == i] = um.ids[name]

#### Add ores

In [ ]:
for name, i in om.ids.items():
    if i != 0:
        mask = om.assignments == i
        mask[um.assignments == um.ids["bedrock"]] = False
        um.assignments[mask] = um.ids[name]

#### Add caves

In [ ]:
cp = CHUNK_PADDING
assignments = cm.assignments[cp:-cp, cp:-cp, cp:-cp]

mask = np.logical_and(
    assignments == cm.ids["cave"],
    um.assignments != um.ids["bedrock"],
)
um.assignments[mask] = 0

#### Plot the distribution

In [ ]:
print("Per voxel:")
for name, i in um.ids.items():
    print(f"{name}: {(um.assignments == i).mean()}")
    
print("\nPer column:")
for name, i in um.ids.items():
    print(f"{name}: {(um.assignments == i).sum(axis=1).mean()}")

#### Plot the voxels

In [ ]:
visualize_voxels(np.array(um.colors)[um.assignments])

### Save the underground map

In [ ]:
chunk_name = "_".join([str(d) for d in np.array(CHUNK_ORIGIN) // np.array(CHUNK_DIM)])

In [ ]:
%%time

with gzip.open(f"F:/Biomes/alpha_world/underground/um_{chunk_name}.dump", "wb") as f:
    pickle.dump(um, f)